# Feature Selection

**A Theoretical & Practical Lecture — Using the used_cars_dataset.csv**


Dataset columns: Car_ID, Brand, Model, Year, Age_Years, Price, Kilometer, Fuel_Type, Transmission, Owner_Type, Mileage_kmpl, Engine_CC, Power_BHP, Seats, Color, City

Throughout this lecture we'll use this exact dataset as our running example, so every technique is demonstrated on a realistic, relatable problem: which of these 15 columns actually help predict a used car's price, and which are noise?

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("used_cars_dataset.csv")
print(df.shape)
df.head()

**Introduction to Feature Selection**

Feature selection is the process of choosing the subset of input columns that actually help predict the target — here, Price — while discarding columns that add noise, redundancy, or no signal at all.

It is not the same as feature extraction (like PCA). We are not creating new combined variables; we are deciding, from the 15 original columns, which ones earn a place in the model.

Why Does It Matter for This Dataset Specifically?

Looking at the columns already raises a few questions a good analyst should ask before modeling:

Does Car_ID (row 1, 2, 3…) tell us anything about Price? Clearly not — it's just a row label.

Year and Age_Years look like they describe the same thing in two different ways. Do we need both?

Does Color (Green, White, Brown, Red…) really move the price of a used car much? Maybe a little, maybe not at all.

Engine_CC and Power_BHP are both about engine strength — are they redundant?

Feature selection gives us a principled way to answer these questions with evidence instead of guesswork.

**Removing Irrelevant Features**

**What to Remove From This Dataset**

Car_ID — a pure row identifier. It has zero relationship to Price by construction and including it risks the model "memorizing" row order instead of learning real patterns.

Potential leakage check — none of the remaining columns (Brand, Model, Year, Kilometer, Fuel_Type, etc.) describe information that would only be known after the sale, so there's no leakage here — good.

High-cardinality text columns to watch — Model has many unique values (X5, Picatto, Santro, Accord, Golf...). It's not irrelevant, but it may need grouping/encoding carefully rather than being dropped outright (see §7).

In [ ]:
# Drop the pure identifier column
df = df.drop(columns=["Car_ID"])

# Sanity check: any fully duplicated columns?
duplicate_cols = df.columns[df.T.duplicated()].tolist()
print("Duplicate columns:", duplicate_cols)   # expect: []

print("Remaining shape:", df.shape)

**Correlation Analysis**
Theory

We check two things:

Feature vs. Price — which numeric columns move together with price?

Feature vs. Feature — do any columns carry duplicate information (multicollinearity)?

The Year / Age_Years Problem

This dataset has a textbook multicollinearity case: Year (e.g., 2019) and Age_Years (e.g., 7) are almost perfectly negatively correlated — they encode the same fact two different ways (Age_Years ≈ current_year − Year). Keeping both adds no new information and can destabilize a linear model's coefficients. We should keep one, not both.

In [ ]:
numeric_cols = ["Year", "Age_Years", "Price", "Kilometer",
                 "Mileage_kmpl", "Engine_CC", "Power_BHP", "Seats"]

corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap — Used Cars Dataset")
plt.show()

# Correlation of each numeric feature with the target, Price
print(corr_matrix["Price"].sort_values(ascending=False))

# Detect redundant feature pairs (|r| > 0.9)
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
redundant_pairs = [(col, row) for col in upper.columns for row in upper.index
                    if pd.notna(upper.loc[row, col]) and abs(upper.loc[row, col]) > 0.9]
print("Highly correlated pairs:", redundant_pairs)
# Expect to see (Year, Age_Years) flagged here

# Drop one of the redundant pair — keep Age_Years, it's more directly interpretable
df = df.drop(columns=["Year"])

**Expected Reasoning on Signal Strength**

1) Age_Years and Kilometer are typically negatively correlated with Price — older, higher-mileage cars sell for less.

2) Power_BHP and Engine_CC are typically positively correlated with Price — more powerful cars cost more.

3) Seats will likely show a weak correlation with Price — most cars in the dataset cluster around 5 seats, so it doesn't vary much (this connects directly to §4).

**Variance Threshold**

A feature with almost no variance carries almost no information, because it barely changes across cars.

Look at the Seats column in the screenshot: nearly every row shows 5. If 95%+ of cars in the dataset have exactly 5 seats, this column is close to constant — it will rarely help distinguish a Rs. 300,000 car from a Rs. 13,000,000 car.

Caution: *Variance depends on scale. Kilometer naturally has huge raw variance (tens of thousands) compared to Seats (a handful of integers) — this doesn't mean Kilometer is more informative, just larger in scale. Always scale numeric features to [0,1] before comparing variances directly.*

In [ ]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import MinMaxScaler

numeric_features = ["Age_Years", "Kilometer", "Mileage_kmpl",
                     "Engine_CC", "Power_BHP", "Seats"]

X_num = df[numeric_features]
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_num), columns=numeric_features)

# Check raw variance first (diagnostic, not the final decision)
print(X_scaled.var().sort_values())

selector = VarianceThreshold(threshold=0.01)
selector.fit(X_scaled)

kept = X_num.columns[selector.get_support()]
dropped = X_num.columns[~selector.get_support()]
print("Kept:", list(kept))
print("Dropped (near-constant):", list(dropped))
# Seats is a strong candidate to be flagged here if it's ~constant at 5

**Univariate Feature Selection**

Since Price is a continuous target, we use regression-appropriate univariate tests instead of the classification versions:

For categorical columns (Brand, Fuel_Type, Transmission, Owner_Type, City, Color), we first need to encode them numerically (e.g., one-hot or target encoding) before scoring, since these tests require numeric input.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression

# --- Numeric features ---
X_numeric = df[["Age_Years", "Kilometer", "Mileage_kmpl",
                  "Engine_CC", "Power_BHP", "Seats"]]
y = df["Price"]

f_scores, _ = f_regression(X_numeric, y)
mi_scores = mutual_info_regression(X_numeric, y, random_state=42)

scores_df = pd.DataFrame({
    "feature": X_numeric.columns,
    "f_score": f_scores,
    "mi_score": mi_scores
}).sort_values("f_score", ascending=False)

print(scores_df)
# Expect Power_BHP, Engine_CC, Age_Years, Kilometer near the top;
# Seats likely near the bottom given its near-constant nature

# --- Categorical features: one-hot encode, then score ---
cat_cols = ["Brand", "Fuel_Type", "Transmission", "Owner_Type", "City"]
X_cat = pd.get_dummies(df[cat_cols], drop_first=True)

selector_cat = SelectKBest(score_func=f_regression, k=10)
selector_cat.fit(X_cat, y)
top_cat_features = X_cat.columns[selector_cat.get_support()]
print("Top encoded categorical features:", list(top_cat_features))
# Watch which Brand_ or Fuel_Type_ dummy columns surface as significant

**Recursive Feature Elimination (RFE)**

RFE trains a model on all features, ranks them by the model's own coefficients/importances, drops the weakest, and repeats — until the target number of features remains. Unlike univariate tests, RFE considers combinations of features (e.g., it can notice that Engine_CC and Power_BHP overlap in what they explain, and only keep the stronger of the two).

RFECV automates how many features to keep, using cross-validation instead of a fixed guess.

In [ ]:
from sklearn.feature_selection import RFE, RFECV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold

# Build a full modeling feature matrix: numeric + one-hot categorical
X_full = pd.concat([
    df[["Age_Years", "Kilometer", "Mileage_kmpl", "Engine_CC", "Power_BHP", "Seats"]],
    pd.get_dummies(df[["Brand", "Fuel_Type", "Transmission", "Owner_Type", "City", "Color"]],
                    drop_first=True)
], axis=1)
y = df["Price"]

# --- RFE with a fixed number of features ---
model = LinearRegression()
rfe = RFE(estimator=model, n_features_to_select=10)
rfe.fit(X_full, y)

selected_rfe = X_full.columns[rfe.support_]
print("RFE selected features:", list(selected_rfe))

# --- RFECV: let cross-validation pick the best number ---
cv = KFold(n_splits=5, shuffle=True, random_state=42)
rfecv = RFECV(
    estimator=RandomForestRegressor(n_estimators=200, random_state=42),
    step=1,
    cv=cv,
    scoring="r2",
    min_features_to_select=5
)
rfecv.fit(X_full, y)

print("Optimal number of features:", rfecv.n_features_)
print("Selected features:", list(X_full.columns[rfecv.support_]))

plt.figure(figsize=(8, 5))
plt.plot(range(5, len(rfecv.cv_results_["mean_test_score"]) + 5),
          rfecv.cv_results_["mean_test_score"])
plt.xlabel("Number of features selected")
plt.ylabel("Cross-validated R²")
plt.title("RFECV Performance — Used Cars Price Prediction")
plt.show()

**Feature Importance**

1) Tree-based importance (Random Forest) will show how much each feature reduces prediction error across all splits — expect Power_BHP, Engine_CC, Age_Years, and Kilometer to dominate, since these directly relate to a car's mechanical value and depreciation.

2) Permutation importance shuffles one column at a time (e.g., shuffle all Brand values across rows) and measures how much R² drops — this avoids the bias tree importance has toward high-cardinality columns like Model or City, which have many unique categories and can appear "important" just by having more splits available.

3) Lasso (L1) regression will shrink weak features — like individual Color dummy variables — to exactly zero.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

# --- Tree-based importance ---
rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_full, y)

importances = pd.Series(rf.feature_importances_, index=X_full.columns).sort_values(ascending=False)
print("Top 10 by Random Forest importance:")
print(importances.head(10))

importances.head(15).plot(kind="barh", figsize=(8, 6))
plt.gca().invert_yaxis()
plt.title("Random Forest Feature Importance — Used Car Price")
plt.show()

# --- Permutation importance (model-agnostic, fairer to Model/Brand/City) ---
perm_result = permutation_importance(rf, X_full, y, n_repeats=10, random_state=42, n_jobs=-1)
perm_importances = pd.Series(perm_result.importances_mean, index=X_full.columns).sort_values(ascending=False)
print("Top 10 by Permutation importance:")
print(perm_importances.head(10))

# --- Lasso: which categorical dummies survive? ---
X_scaled = StandardScaler().fit_transform(X_full)
lasso = Lasso(alpha=1000)   # alpha tuned relative to Price's scale (hundreds of thousands)
lasso.fit(X_scaled, y)

lasso_coefs = pd.Series(lasso.coef_, index=X_full.columns)
selected_by_lasso = lasso_coefs[lasso_coefs != 0].sort_values(key=abs, ascending=False)
print("Non-zero Lasso coefficients:")
print(selected_by_lasso)
# Watch whether Color_* dummies survive — they are the most likely to be zeroed out

**Selecting the Final Features**

| **Feature**      | **Likely Verdict** | **Reason / Justification**                                                                                                                          |
| ---------------- | ------------------ | --------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Car_ID**       | ❌ Drop             | Pure identifier with no meaningful predictive value.                                                                                                |
| **Year**         | ❌ Drop             | Redundant with **Age_Years**, which provides the same information in a more interpretable form.                                                     |
| **Age_Years**    | ✅ Keep             | Strong and interpretable driver of vehicle depreciation.                                                                                            |
| **Kilometer**    | ✅ Keep             | Typically has a strong negative relationship with resale price; higher mileage generally reduces vehicle value.                                     |
| **Engine_CC**    | ✅ Keep             | Important mechanical characteristic that can influence vehicle price.                                                                               |
| **Power_BHP**    | ✅ Keep             | Strong mechanical value driver; higher power may contribute to higher resale value.                                                                 |
| **Mileage_kmpl** | ✅ Likely Keep      | Fuel efficiency can significantly affect buyer demand and resale value.                                                                             |
| **Seats**        | ❌ Likely Drop      | Near-constant feature with little variation, providing limited predictive information.                                                              |
| **Fuel_Type**    | ✅ Keep (Encoded)   | Fuel type can meaningfully affect resale value and buyer preferences.                                                                               |
| **Transmission** | ✅ Keep (Encoded)   | Manual vs. automatic transmission can have a significant impact on resale value.                                                                    |
| **Owner_Type**   | ✅ Keep (Encoded)   | Number/type of previous owners can influence buyer perception and resale price.                                                                     |
| **Brand**        | ✅ Keep with Care   | Strong predictive signal, but high cardinality may require grouping rare categories.                                                                |
| **Model**        | ✅ Keep with Care   | Often strongly associated with price, but high cardinality requires careful encoding or grouping of rare models.                                    |
| **City**         | ⚠️ Test Carefully  | May capture regional differences in vehicle prices, but could also introduce noise. Let RFE or feature-importance methods determine its usefulness. |
| **Color**        | ❌ Likely Drop      | Generally has a weak relationship with used-car prices in most markets.                                                                             |
